# 122 — Evaluación y depuración de agentes

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Dos ejes complementarios:

- **Eval de resultado:** ¿el estado final satisface el objetivo? (predicado
  ejecutable: tests pasan, archivo válido). Es lo que importa al usuario; ignora el
  camino.
- **Eval de proceso:** ¿el CÓMO fue correcto? — tools pertinentes, argumentos
  válidos, permisos respetados, presupuesto razonable. Detecta éxitos por casualidad
  (✓ resultado, ✗ proceso: fallará pronto) y fallos por una decisión reparable.

La matriz 2×2 resultado×proceso es la primera herramienta diagnóstica. Métricas sobre
un conjunto de tareas reproducible: tasa de éxito (con su varianza), pass@k, costo
por éxito, pasos vs óptimo, violaciones. LLM-as-judge para criterios blandos — con
rúbrica y calibración contra humanos.

### 🔬 Depurar = leer trayectorias

Método: (1) reunir trayectorias FALLIDAS del eval; (2) localizar el **primer paso
divergente** (la primera decisión que un experto no tomaría — el fallo visible suele
ser síntoma posterior); (3) clasificar la causa raíz (E1 instrucciones, E2 selección
de tool, E3 argumentos, E4 interpretación de la observación, E5 planificación,
E6 parada, E7 entorno/tool); (4) CONTAR y arreglar la categoría dominante;
(5) re-ejecutar el eval completo — sin re-ejecución no hay evidencia de mejora ni
detección de regresiones.

El laboratorio `evaluation` entrega la matriz mínima: tp=3, fp=1, fn=1 →
precision = recall = 0,75, con la advertencia honesta de que 8 ejemplos no estiman
desempeño real.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** precision = 3/(3+1) = 0,75; recall = 3/(3+1) = 0,75;
F1 = 2·0,75·0,75/1,5 = 0,75. Con FN 10× más caro, se vigila el **recall** (cada FN
que se escapa cuesta como 10 FP) y probablemente se baja el umbral del detector
aceptando peor precision — la métrica se elige con el costo de error, no por
costumbre.

**Ejercicio 2.** Matriz: ✓✓ = 6; ✓✗ = 2 (test borrado + redundancia); ✗✓ = 3 (E7:
entorno — la API falló, el proceso era correcto); ✗✗ = 1 (E5: planificación). Tasa
ingenua = 8/12 = 67 %; honesta (✓✓) = 6/12 = 50 %. Los ✓✗ son los más urgentes:
premiados por la métrica ingenua y prohibibles por permisos (119); para E7,
reintento con backoff (113/118), no cambios de prompt.

**Ejercicio 3.** Primer paso divergente: el **paso 2** (buscar en `api/` cuando el
reporte dice `auth/`) — categoría E4 (mala interpretación de la observación del paso
1) o E2 si la tool de búsqueda tiene descripción ambigua. Los pasos 3-5 son
consecuencia: editar sobre diagnóstico erróneo y repetir la edición (síntoma E6 de no
replantear). Intervención: exigir en el prompt/plan que la localización cite
literalmente el módulo del reporte antes de editar (hito verificable de la 112). El
paso 5 no es causa: sin arreglar el paso 2, prohibir re-ediciones solo cambiaría el
modo de fallo.

**Ejercicio 4.** Ver celda: la tasa global sube de 60 % a 70 % pero las tareas t3 y
t7 se rompen. Se despliega solo si las regresiones se explican y aceptan
explícitamente — las mejoras medias con regresiones concentradas (p. ej. en la
categoría más crítica) son la forma clásica de degradar producción con un eval en
verde.

In [ ]:
result = run_lab("evaluation", seed=122)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — métricas verificadas contra el laboratorio
result = run_lab("evaluation", seed=122)
r = result["result"]
tp, fp, fn = r["tp"], r["fp"], r["fn"]
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
assert precision == r["precision"] and recall == r["recall"]
print(f"precision={precision:.3f} recall={recall:.3f} f1={f1:.3f}")
print("FN 10x FP -> vigilar RECALL y bajar umbral aceptando mas FP")


In [ ]:
# Ejercicio 4 — detector de regresión demostrado
def comparar_evals(antes, despues):
    a = {t["id"]: t for t in antes}
    d = {t["id"]: t for t in despues}
    ids = sorted(a)
    tasa_a = sum(a[i]["exito"] for i in ids) / len(ids)
    tasa_d = sum(d[i]["exito"] for i in ids) / len(ids)
    regresiones = [i for i in ids if a[i]["exito"] and not d[i]["exito"]]
    mejoras = [i for i in ids if not a[i]["exito"] and d[i]["exito"]]
    def costo_por_exito(res):
        exitos = sum(t["exito"] for t in res)
        return sum(t["costo"] for t in res) / exitos if exitos else float("inf")
    return {"delta_tasa": tasa_d - tasa_a,
            "regresiones": regresiones, "mejoras": mejoras,
            "costo_por_exito": (costo_por_exito(antes), costo_por_exito(despues))}

antes = [{"id": f"t{i}", "exito": i not in (2, 5, 8, 9), "costo": 0.30} for i in range(10)]
despues = [{"id": f"t{i}", "exito": i not in (3, 7, 9), "costo": 0.25} for i in range(10)]
rep = comparar_evals(antes, despues)
print(rep)
assert rep["delta_tasa"] > 0 and rep["regresiones"] == ["t3", "t7"]
print("la tasa MEJORA pero t3 y t7 se rompen -> revisar antes de desplegar")


## Reflexión

1. Un agente "arregló" el build borrando el test que fallaba: resultado ✓, proceso ✗.
   ¿Qué combinación de eval de proceso (122) y permisos (119) convierte ese caso en
   fallo visible, y por qué la tasa de éxito ingenua lo premiaba?
2. ¿Por qué el "primer paso divergente" es mejor unidad de diagnóstico que el paso
   donde el agente se rindió?
3. El laboratorio declara "ocho ejemplos no estiman desempeño real". ¿Qué decisión
   podrías tomar igualmente con esos 8 ejemplos y cuál sería irresponsable tomar?